# Open-weight unlearning arm — Colab runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wrgr/socratic-scenarios/blob/main/experiments/unlearning/colab.ipynb)

Runs **Experiment 2** of the corpus-bounded-instruction paper: unlearn the
*alter-to-starboard* knowledge (COLREG Rule 14/15) from an open-weight LLM with
**SimNPO**, audit the removal, then score the base vs unlearned model on the
reference-optimal instrument — the 2×2 in `docs/novelty-and-positioning.md` §8.

### Before you run
1. **Runtime → Change runtime type → GPU.** An **A100** or **L4 (High-RAM)** is
   recommended for a 7–8B model in bf16 (~15 GB). On a **T4 (16 GB)** use a smaller
   model (set `MODEL = "Qwen/Qwen2.5-3B-Instruct"` below) or it may OOM.
2. Run the cells top to bottom.

Everything runs in this one Colab runtime — no local machine, API key, or server. The model
steps run on the GPU (Python); scoring runs the repo's TypeScript instrument (Node is
pre-installed on Colab) over a **saved transcript** — offline, no HTTP port.


## 1 · Confirm the GPU


In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime → Change runtime type → GPU'

## 2 · Config + clone the repo

Pick `MODEL`; the notebook auto-selects 4-bit vs bf16 to fit the GPU and threads it
through the whole pipeline. Default is **Qwen2.5-3B** (bf16, fits a T4 with room). For the
bigger **Qwen2.5-7B** set `MODEL` to it — 4-bit QLoRA turns on automatically. `BRANCH`
defaults to `main`; point it at a tag or commit SHA if you need a frozen revision.
Re-running pulls the latest.

In [ ]:
import os

REPO_URL = 'https://github.com/wrgr/socratic-scenarios.git'
BRANCH   = 'main'          # or a tag / commit SHA for a frozen revision

# ── Pick a model. Two profiles tested on a 16 GB T4 — the notebook handles either: ──
#   'Qwen/Qwen2.5-3B-Instruct'  → bf16, no 4-bit   (roomy, simplest — the default)
#   'Qwen/Qwen2.5-7B-Instruct'  → 4-bit QLoRA       (bigger model; auto-enables bitsandbytes)
MODEL   = 'Qwen/Qwen2.5-3B-Instruct'
METHOD  = 'simnpo'         # simnpo (primary, reference-free = lighter) | npo | ga
SEED    = '0'              # change + re-run to measure variance (report >=3 seeds)
RELEARN = '0'              # '1' also runs the benign-relearning 'gone vs suppressed' test
LR      = '1e-4'           # LOWER (e.g. '5e-5') if unlearned output degrades (garbled/Chinese)
RETAIN_WEIGHT = '1.0'      # RAISE (e.g. '3.0') to protect fluency / retain knowledge

# Auto-config: a 7-8B on a T4 needs 4-bit NF4; 3B and below run in plain bf16. The whole
# pipeline (unlearn/audit/score) reads LOAD_4BIT, so switching MODEL is the only change.
DTYPE     = 'bfloat16'     # compute dtype on GPU; float32 only on CPU
LOAD_4BIT = '1' if any(s in MODEL for s in ('7B', '8B', '13B', '14B')) else '0'

%cd /content
# Clone if absent, then always hard-sync to the branch tip so a re-run never runs stale code
# (robust on a shallow clone; discards only the throwaway clone, not your Colab edits).
![ -d socratic-scenarios ] || git clone --depth 1 --branch $BRANCH $REPO_URL
!cd socratic-scenarios && git fetch --depth 1 origin $BRANCH && git reset --hard FETCH_HEAD
REPO = '/content/socratic-scenarios'
ARM  = REPO + '/experiments/unlearning'
%cd $ARM
print(f'resolved: model={MODEL}  method={METHOD}  dtype={DTYPE}  load_4bit={LOAD_4BIT}  seed={SEED}  relearn={RELEARN}  lr={LR}  retain_w={RETAIN_WEIGHT}')

## 3 · Install Python deps

Colab GPU runtimes already ship a CUDA build of `torch`; we add the HF stack only.


In [ ]:
!pip -q install 'transformers>=4.40' 'peft>=0.11' 'accelerate>=0.30' 'safetensors>=0.4' 'bitsandbytes>=0.43'
# Colab ships an old torchao (0.10) that PEFT's LoRA dispatch rejects with an
# ImportError (it wants torchao>=0.16). We don't use torchao — remove it so PEFT
# falls back to the plain Linear LoRA path. (Alternatively: pip install -U torchao.)
!pip -q uninstall -y torchao 2>/dev/null; echo removed-torchao-if-present
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
# bitsandbytes is only needed when LOAD_4BIT=1; the import is lazy in the scripts.

## 4 · Run the arm — build → unlearn (SimNPO) → audit

`run.sh` builds the forget/retain/audit sets, LoRA-unlearns the target rule, and prints the
removal audit: forget-set NLL ↑, retain-set NLL + **coherence**, and each forget probe
classified survived / wrong-direction / degenerate / abstained — with a **survived-rate by
probe type** (direct vs paraphrase / jailbreak / indirect; high on non-direct ⇒ suppressed,
not gone). With `RELEARN='1'` it also runs the benign-relearning test (does a few steps of
fine-tuning bring the behavior back — *suppressed* vs *removed*). A 7–8B run is
~minutes–hours depending on the GPU.


In [ ]:
import os
# Runs build -> unlearn -> audit; tees the output to a file for the consolidated report.
!cd $ARM && MODEL=$MODEL METHOD=$METHOD DTYPE=$DTYPE LOAD_4BIT=$LOAD_4BIT SEED=$SEED RELEARN=$RELEARN LR=$LR RETAIN_WEIGHT=$RETAIN_WEIGHT bash run.sh 2>&1 | tee /content/arm-audit.txt

## 5 · Score base vs unlearned on the instrument (portless)

> **Do I need an API key here? No.** This arm scores the **local** open-weight model you
> just unlearned. No Gemini or OpenAI key is used or needed anywhere in this notebook.

Scoring is **offline — no server, no port.** The leakage/diagnosis prompt set is fully
determined by the static scenario/corpus config (the scorer never branches on model output),
so we split scoring into three portless phases:

1. **Dump** the exact prompt set once (`LEAKAGE_DUMP` → `prompts.jsonl`).
2. **Generate** completions with the model — base (adapter off) and unlearned (adapter on)
   — via `score_offline.py`, writing a saved `{prompt, completion}` transcript.
3. **Replay** each transcript through the instrument (`LEAKAGE_REPLAY`).

A single scoring run already reports the metric **with** the rule in the corpus
(`metricWith`) and **without** it (`metricWithout`, ablated) — those are the two columns of
the §8 2×2 — plus the closed-book (no-corpus) baseline. So one transcript per model covers
its whole row. Read the two reports as the 2×2: the **base** model should turn starboard
from pretrained priors even closed-book (contamination baseline); the **unlearned** model
should fail the governed metric without the corpus and recover with it. The transcripts are
saved (`completions-*.jsonl`) — a reproducible artifact you can re-score without the GPU.


In [ ]:
# One-time: install Node deps for the tsx scorer (npx tsx scripts/colreg-leakage.ts).
# The scorer is TypeScript; the model runs in Python — offline transcripts bridge them.
!cd $REPO && npm install --no-audit --no-fund --loglevel=error

In [ ]:
import os, subprocess

DUMP = os.path.join(ARM, 'prompts.jsonl')

# Phase 1 - dump the deterministic prompt set once (identical for base and unlearned).
subprocess.run(['npm', 'run', 'colreg:leakage'], cwd=REPO, check=True,
               env=dict(os.environ, LEAKAGE_DUMP=DUMP))

def score(label, adapter):
    trans = os.path.join(ARM, f'completions-{label}.jsonl')
    gen = ['python', 'score_offline.py', '--model', MODEL, '--dtype', DTYPE,
           '--prompts', DUMP, '--out', trans]
    if LOAD_4BIT == '1':
        gen += ['--load_4bit']
    if adapter:
        gen += ['--adapter', adapter]
    print(f'\n===== {label}: generating completions =====')
    subprocess.run(gen, cwd=ARM, check=True)
    print(f'===== {label}: scoring =====')
    r = subprocess.run(['npm', 'run', 'colreg:leakage'], cwd=REPO, capture_output=True, text=True,
                       env=dict(os.environ, LEAKAGE_REPLAY=trans))
    open(f'/content/leakage-{label}.txt', 'w').write(r.stdout)   # saved for the report
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print('--- stderr ---'); print(r.stderr[-2000:])

score('base', None)                       # base row of the 2x2 (adapter off)
score('unlearned', 'out/unlearned')       # unlearned row of the 2x2 (adapter on)

## 6 - Single copy-paste report

Run the next cell, then **select all of its output and paste it back** - it gathers *everything* (config, removal audit, the 2x2 for base and unlearned, and the unlearned completions so you can eyeball inversion/coherence) into one block. It also zips the full artifacts to `/content/unlearning-results.zip`.

In [ ]:
import os, json, glob, shutil

def _read(path):
    return open(path).read().strip() if os.path.exists(path) else f'(missing: {path})'

def _audit_tail(path):
    if not os.path.exists(path): return '(missing)'
    t = open(path).read(); m = t.find('=== BASE (not unlearned) ===')
    return (t[m:] if m >= 0 else t[-3000:]).strip()

def _preview(path, n=8):
    if not os.path.exists(path): return '(missing)'
    return '\n'.join('  * ' + repr(json.loads(l).get('completion', ''))[:130]
                     for l in list(open(path))[:n])

cfg = os.path.join(ARM, 'out/unlearned/unlearn_config.json')
print('==================== UNLEARNING RUN REPORT (copy from here) ====================')
print('\n----- CONFIG (hyperparameters) -----\n' + _read(cfg))
print('\n----- REMOVAL AUDIT -----\n' + _audit_tail('/content/arm-audit.txt'))
print('\n----- 2x2 . BASE -----\n' + _read('/content/leakage-base.txt'))
print('\n----- 2x2 . UNLEARNED -----\n' + _read('/content/leakage-unlearned.txt'))
print('\n----- UNLEARNED completions (coherence / inversion check) -----\n'
      + _preview(os.path.join(ARM, 'completions-unlearned.jsonl')))
print('\n----- BASE completions -----\n'
      + _preview(os.path.join(ARM, 'completions-base.jsonl')))
print('\n==================== END REPORT ====================')

# Also bundle everything for download.
bundle = '/content/unlearning-results'; os.makedirs(bundle, exist_ok=True)
for f in (glob.glob(f'{ARM}/completions-*.jsonl') + glob.glob(f'{ARM}/prompts.jsonl')
          + glob.glob(f'{ARM}/out/*/unlearn_config.json')
          + glob.glob('/content/arm-audit.txt') + glob.glob('/content/leakage-*.txt')):
    shutil.copy(f, bundle)
shutil.make_archive('/content/unlearning-results', 'zip', bundle)
print('\n(Full artifacts also zipped to /content/unlearning-results.zip - Files panel to download.)')

## Done

That's the full arm: unlearn → audit → score base-vs-unlearned on the instrument. Read the
two reports as the §8 2×2 (each run's `metricWith` vs `metricWithout` are the corpus /
no-corpus columns). No API key and **no local server or port** were needed — the model runs
offline in Python and the TypeScript instrument scores a saved transcript. The
`completions-*.jsonl` files are a reproducible artifact you can re-score on CPU.
